# 02 — Linear Regression Bridge

**Purpose:** step from a single-parameter model (coin bias) to a multi-parameter,
continuous model with measurement noise — a Bayesian linear regression. This is
structurally much closer to the IDM calibration problem:

- Multiple unknown parameters estimated jointly (here: slope + intercept; later:
  v0, T, a_max, b, s0)
- A continuous noise model (`Normal` likelihood) instead of `Binomial`
- Priors that encode physical plausibility (e.g. we might know the slope should
  be roughly positive) — the same role priors play for IDM (e.g. `s0` must be
  positive, and its plausible range is a few meters, not kilometers)
- Diagnosing when posteriors are well- vs. poorly-constrained by the data —
  building directly toward the free-flow / congested identifiability comparison
  for IDM's `s0`.


In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


## 1. Generate synthetic linear data

`y = alpha + beta * x + noise`, with known true `alpha`, `beta`, `sigma`.

In [ ]:
alpha_true, beta_true, sigma_true = 2.0, 3.5, 1.2
n = 60

x = rng.uniform(-3, 3, n)
y = alpha_true + beta_true * x + rng.normal(0, sigma_true, n)

plt.scatter(x, y, alpha=0.7)
plt.xlabel("x"); plt.ylabel("y")
plt.title("Synthetic data")
plt.show()


## 2. Bayesian linear regression model

Priors are weakly informative Normals on `alpha`/`beta`, and a `HalfNormal` on
noise `sigma` (must be positive) — same pattern we'll reuse for IDM's strictly
positive parameters (v0, a_max, b, s0 all need positivity constraints, handled
there with `TruncatedNormal`).

In [ ]:
with pm.Model() as lin_model:
    alpha = pm.Normal("alpha", mu=0, sigma=10)
    beta = pm.Normal("beta", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)

    mu = alpha + beta * x
    pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

    idata_lin = pm.sample(2000, tune=1000, chains=4, random_seed=RANDOM_SEED)

az.summary(idata_lin, var_names=["alpha", "beta", "sigma"])


In [ ]:
az.plot_posterior(idata_lin, var_names=["alpha", "beta", "sigma"],
                   ref_val=[alpha_true, beta_true, sigma_true])
plt.show()


## 3. Posterior predictive check

A key habit to carry forward: don't just look at parameter posteriors — check
whether the fitted model actually reproduces data that *looks like* what was
observed. We'll do the equivalent for IDM (predicted vs. observed acceleration)
in notebook 03.

In [ ]:
with lin_model:
    ppc = pm.sample_posterior_predictive(idata_lin, random_seed=RANDOM_SEED)

y_pred_samples = ppc.posterior_predictive["y_obs"].values.reshape(-1, n)
y_pred_mean = y_pred_samples.mean(axis=0)

order = np.argsort(x)
plt.scatter(x, y, alpha=0.5, label="observed")
plt.plot(x[order], y_pred_mean[order], color="red", label="posterior predictive mean")
plt.legend(); plt.title("Posterior predictive check")
plt.show()


## 4. Identifiability preview: what happens with collinear / uninformative data?

If `x` barely varies, the slope `beta` becomes hard to pin down — the posterior
stays wide even with lots of data points, because the *data itself* doesn't
constrain the slope. This is conceptually identical to what happens with IDM's
`s0` in free-flowing traffic: when vehicles never get close enough to each other
for the minimum-gap term to bind, no amount of data will tightly constrain it.

In [ ]:
x_narrow = rng.uniform(-0.05, 0.05, n)   # almost no variation in x
y_narrow = alpha_true + beta_true * x_narrow + rng.normal(0, sigma_true, n)

with pm.Model():
    alpha_n = pm.Normal("alpha", mu=0, sigma=10)
    beta_n = pm.Normal("beta", mu=0, sigma=10)
    sigma_n = pm.HalfNormal("sigma", sigma=5)
    pm.Normal("y_obs", mu=alpha_n + beta_n * x_narrow, sigma=sigma_n, observed=y_narrow)
    idata_narrow = pm.sample(2000, tune=1000, chains=4, progressbar=False, random_seed=RANDOM_SEED)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(idata_lin.posterior["beta"].values.flatten(), bins=40, alpha=0.7)
axes[0].axvline(beta_true, color="red", linestyle="--")
axes[0].set_title("beta posterior: x well-spread")

axes[1].hist(idata_narrow.posterior["beta"].values.flatten(), bins=40, alpha=0.7, color="tab:orange")
axes[1].axvline(beta_true, color="red", linestyle="--")
axes[1].set_title("beta posterior: x barely varies\n(much wider -> weakly identified)")
fig.tight_layout()
plt.show()


## Takeaways -> carried into the IDM notebook

1. Multi-parameter Bayesian models follow the same recipe as the coin: priors +
   likelihood -> posterior via MCMC.
2. Posterior predictive checks verify the fitted model actually reproduces
   observed behavior, not just "looks reasonable on paper."
3. A parameter's posterior width reflects how much the *specific dataset*
   constrains it — a classical point estimate would report a single `beta`
   number in both the well-spread and narrow-x cases, hiding the fact that the
   second one is barely informative. This exact pattern reappears with IDM's
   `s0` in `03_idm_bayesian_calibration.ipynb`.
